# 郵便番号API

In [ ]:
import requests

# 郵便番号を指定する（さいたま市のど真ん中で試してみます）
zipcode = "3500043"  # 川越市のあたり

# APIにリクエストを送る
url = f"https://zipcloud.ibsnet.co.jp/api/search?zipcode={zipcode}"
response = requests.get(url)

# 結果を表示する
# **ステータスコード**
# ```
# 200 → 成功
# 404 → 見つからない
# 500 → サーバーエラー
print(f"ステータスコード：{response.status_code}")
print(f"取得したデータ：")
print(response.json())

ステータスコード：200
取得したデータ：
{'message': None, 'results': [{'address1': '埼玉県', 'address2': '川越市', 'address3': '新富町', 'kana1': 'ｻｲﾀﾏｹﾝ', 'kana2': 'ｶﾜｺﾞｴｼ', 'kana3': 'ｼﾝﾄﾐﾁｮｳ', 'prefcode': '11', 'zipcode': '3500043'}], 'status': 200}


In [ ]:
# 取得したデータを見やすく整理する
data = response.json()
result = data["results"][0]

print(f"郵便番号：{result['zipcode']}")
print(f"住所　　：{result['address1']}{result['address2']}{result['address3']}")
print(f"ふりがな：{result['kana1']}{result['kana2']}{result['kana3']}")

# response.json()　　　# 取得したデータをPythonで扱える形に変換
# data["results"]　　　# resultsというキーのデータを取り出す
# [0]　　　　　　　　　# 最初の1件目を取り出す（リストの0番目）
# result["address1"]　 # address1というキーの値を取り出す

郵便番号：3500043
住所　　：埼玉県川越市新富町
ふりがな：ｻｲﾀﾏｹﾝｶﾜｺﾞｴｼｼﾝﾄﾐﾁｮｳ


In [ ]:
import pandas as pd
import time

# 従業員の郵便番号リスト
employees = [
    {"氏名": "田中 太郎", "郵便番号": "3500043"},
    {"氏名": "鈴木 花子", "郵便番号": "3300063"},
    {"氏名": "佐藤 次郎", "郵便番号": "1000001"},
    {"氏名": "山田 美咲", "郵便番号": "4600008"},
    {"氏名": "伊藤 健一", "郵便番号": "9999999"},  # 存在しない郵便番号
]

results = []

for emp in employees:
    url = f"https://zipcloud.ibsnet.co.jp/api/search?zipcode={emp['郵便番号']}"
    response = requests.get(url)
    data = response.json()

    if data["results"]:  # 住所が見つかった場合
        r = data["results"][0]
        results.append({
            "氏名":   emp["氏名"],
            "郵便番号": emp["郵便番号"],
            "住所":   f"{r['address1']}{r['address2']}{r['address3']}",
            "ふりがな": f"{r['kana1']}{r['kana2']}{r['kana3']}",
            "状態":   "✅ 取得成功"
        })
    else:  # 住所が見つからなかった場合
        results.append({
            "氏名":   emp["氏名"],
            "郵便番号": emp["郵便番号"],
            "住所":   "",
            "ふりがな": "",
            "状態":   "⚠️ 見つかりません"
        })

    time.sleep(0.5)  # APIに負荷をかけないよう0.5秒待つ

# 結果をDataFrameにする
df_address = pd.DataFrame(results)
print(f"✅ 処理完了：{len(df_address)}件")
df_address

✅ 処理完了：5件


,氏名,郵便番号,住所,ふりがな,状態
0,田中 太郎,3500043,埼玉県川越市新富町,ｻｲﾀﾏｹﾝｶﾜｺﾞｴｼｼﾝﾄﾐﾁｮｳ,✅ 取得成功
1,鈴木 花子,3300063,埼玉県さいたま市浦和区高砂,ｻｲﾀﾏｹﾝｻｲﾀﾏｼｳﾗﾜｸﾀｶｻｺﾞ,✅ 取得成功
2,佐藤 次郎,1000001,東京都千代田区千代田,ﾄｳｷｮｳﾄﾁﾖﾀﾞｸﾁﾖﾀﾞ,✅ 取得成功
3,山田 美咲,4600008,愛知県名古屋市中区栄,ｱｲﾁｹﾝﾅｺﾞﾔｼﾅｶｸｻｶｴ,✅ 取得成功
4,伊藤 健一,9999999,,,⚠️ 見つかりません


In [ ]:
# Googleドライブと連携する→初回、Googleアカウントの認証画面が出るので許可する
from google.colab import drive
drive.mount('/content/drive')

import os #ファイルやディレクトリの作成・削除・名前変更、パス操作、環境変数取得など、OS（オペレーティングシステム）に依存する操作をPythonから簡単に行える標準ライブラリ
from datetime import datetime #datetimeという部品セットの中から、datetimeという名前の**特定の道具（クラス）**だけを直接取り出します。

# 今日の日付と時間をファイル名に追加
now = datetime.now().strftime("%Y%m%d_%H%M%S")
output_path = f"/content/drive/MyDrive/Colab Notebooks/従業員住所録_{now}.xlsx"
# datetime.now()                    # 現在の日時を取得
# .strftime("%Y%m%d_%H%M%S")       # 書式を指定して文字列に変換
# # 例：2024年10月15日 14時30分25秒
# # → "20241015_143025"
# 記号	意味	例
# %Y	西暦4桁	2024
# %m	月2桁	10
# %d	日2桁	15
# %H	時2桁	14
# %M	分2桁	30
# %S	秒2桁	25

# 同じファイル名が存在するか確認
if os.path.exists(output_path):
    print(f"⚠️ 同じファイルが既に存在します：")
    print(f"   {output_path}")
    answer = input("上書き保存しますか？（y / n）：")

    if answer.lower() == "y":
        df_address.to_excel(output_path, index=False, sheet_name="住所録")
        print("✅ 上書き保存しました")
    else:
        print("❌ 保存をキャンセルしました")
else:
    df_address.to_excel(output_path, index=False, sheet_name="住所録")
    print("✅ Excelファイルを書き出しました")
    print(f"ファイル名：従業員住所録_{now}.xlsx")
# os.path.exists(path)  # ファイルが存在するか確認→True/False
# input("質問文")       # ユーザーに入力を求める
# .lower()              # 入力を小文字に統一（Y→y）

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Excelファイルを書き出しました
ファイル名：従業員住所録_20260401_220450.xlsx


In [ ]:
# # Googleドライブと連携する→初回、Googleアカウントの認証画面が出るので許可する
# from google.colab import drive
# drive.mount('/content/drive')

# # Excelファイルを読み込む
# # import pandas as pd
# df = pd.read_excel("/content/drive/MyDrive/Colab Notebooks/勤怠データ_2024年10月.xlsx")
# print(f"読み込み成功：{len(df)}行")

# # 住所データをExcelに書き出す
# output_path = "/content/drive/MyDrive/Colab Notebooks/従業員住所録.xlsx"

# df_address.to_excel(output_path, index=False, sheet_name="住所録")

# print("✅ Excelファイルを書き出しました")
# print(f"保存先：{output_path}")